In [ ]:
import pandas as pd

df = pd.read_csv('grid_search_job00000.csv')
df.head()

In [ ]:
df.columns

In [ ]:
df['HOTA_13d002'].max()

# print the row with the maximum HOTA_13d002 value
row_best_hota_13 = df[df['HOTA_13d002'] == df['HOTA_13d002'].max()]
row_best_hota_31 = df[df['HOTA_31d005'] == df['HOTA_31d005'].max()]
row_best_hota_combined = df[df['HOTA_combined'] == df['HOTA_combined'].max()]
print("Best HOTA_13d002:")
for col in df.columns:
    print(f"{col}: {row_best_hota_13[col].values[0]}")
print("\n")
# line 
print("-*" * 20)
print("Best HOTA_31d005:")
for col in df.columns:
    print(f"{col}: {row_best_hota_31[col].values[0]}")
print("\n")
print("-*" * 20)
print("Best HOTA_combined:")
for col in df.columns:
    print(f"{col}: {row_best_hota_combined[col].values[0]}")
    

In [ ]:
# printing is boring. what's the best way to visualise this?
import matplotlib.pyplot as plt
import seaborn as sns
# create a bar plot of the best HOTA_13d002 and HOTA_31d005 values
sns.barplot(x=['HOTA_13d002', 'HOTA_31d005'], y=[row_best_hota_13['HOTA_13d002'].values[0], row_best_hota_31['HOTA_31d005'].values[0]])
plt.title('Best HOTA Values')   
plt.ylabel('HOTA Value')
plt.show()

# now find a way to visualise the parameters that led to these best values. maybe a radar plot? or a parallel coordinates plot? let's try a parallel coordinates plot.
from pandas.plotting import parallel_coordinates
# create a parallel coordinates plot for the best HOTA_13d002 and HOTA_31d005 rows
best_rows = pd.concat([row_best_hota_13, row_best_hota_31])
parallel_coordinates(best_rows, class_column='HOTA_13d002', cols=[col for col in df.columns if col not in ['HOTA_13d002', 'HOTA_31d005', 'bw_name']]) 
plt.title('Parallel Coordinates Plot for Best HOTA Values')
plt.show()


In [ ]:
df['HOTA_combined_median'] = df[['HOTA_13d002', 'HOTA_31d005']].median(axis=1)

# now find the row with the maximum HOTA_combined_median value
row_best_hota_combined_median = df[df['HOTA_combined_median'] == df['HOTA_combined_median'].max()]
print("Best HOTA_combined_median:")
for col in df.columns:
    print(f"{col}: {row_best_hota_combined_median[col].values[0]}")
    

## Age-dependent parameter analysis

For each numeric parameter, compute its **marginal optimal value** at each age point (13d, 31d):
- Group all configs by that parameter's value
- Take the mean HOTA across all other parameters
- The argmax is the "marginally optimal" value at that age

If the optimal value shifts between ages, we can fit `param_optimal = a × age + b` and use that linear rule at inference time instead of a fixed value.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

AGES = [13, 31]
HOTA_COLS = {'13': 'HOTA_13d002', '31': 'HOTA_31d005'}

# Numeric params only (skip bw_name which is categorical)
numeric_params = [
    'confidence', 'minimum_matching_threshold', 'inertia', 'delta_t',
    'jump_factor', 'overlap_weight_scale', 'overlap_iou_scale',
    'w_under', 'bw_speed', 'bw_turning_angle', 'bw_pause',
    'bw_acceleration', 'bw_tortuosity'
]

def marginal_optimal(df, param, hota_col):
    """For a given parameter, find the value that maximises mean HOTA when we
    average over all other parameter combinations."""
    grouped = df.groupby(param)[hota_col].mean()
    return grouped.idxmax(), grouped

results = {}
for param in numeric_params:
    opt_13, curve_13 = marginal_optimal(df, param, HOTA_COLS['13'])
    opt_31, curve_31 = marginal_optimal(df, param, HOTA_COLS['31'])
    
    # Linear fit: optimal_value = a * age + b  (only 2 points, so exact)
    slope = (opt_31 - opt_13) / (31 - 13)
    intercept = opt_13 - slope * 13
    
    results[param] = {
        'opt_13': opt_13, 'opt_31': opt_31,
        'slope': slope, 'intercept': intercept,
        'delta': opt_31 - opt_13,
        'curve_13': curve_13, 'curve_31': curve_31,
    }

# Sort by absolute delta (most age-sensitive first)
sorted_params = sorted(results, key=lambda p: abs(results[p]['delta']), reverse=True)

print(f"{'Parameter':<30} {'Opt@13d':>8} {'Opt@31d':>8} {'Delta':>8} {'Slope/yr':>10}")
print("-" * 66)
for p in sorted_params:
    r = results[p]
    print(f"{p:<30} {r['opt_13']:>8.3f} {r['opt_31']:>8.3f} {r['delta']:>+8.3f} {r['slope']:>10.4f}")


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

age_x = np.array([13, 31])

for i, param in enumerate(sorted_params):
    ax = axes[i]
    r = results[param]
    
    # HOTA vs param curves at each age
    curve_13 = r['curve_13']
    curve_31 = r['curve_31']
    
    ax2 = ax.twinx()
    ax2.plot(curve_13.index, curve_13.values, 'o-', color='steelblue', alpha=0.5, lw=1.5, ms=4, label='Mean HOTA 13d')
    ax2.plot(curve_31.index, curve_31.values, 's-', color='coral', alpha=0.5, lw=1.5, ms=4, label='Mean HOTA 31d')
    ax2.set_ylabel('Mean HOTA', fontsize=8)
    ax2.tick_params(labelsize=7)
    
    # Mark the optima
    ax2.axvline(r['opt_13'], color='steelblue', lw=1.5, ls='--', alpha=0.8)
    ax2.axvline(r['opt_31'], color='coral', lw=1.5, ls='--', alpha=0.8)
    
    ax.set_title(f"{param}\nΔ={r['delta']:+.3f}  slope={r['slope']:.4f}/d", fontsize=8, pad=3)
    ax.set_xlabel('parameter value', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.set_yticks([])
    
    # Legend only on first plot
    if i == 0:
        lines_13 = plt.Line2D([0], [0], color='steelblue', lw=2, label='13d')
        lines_31 = plt.Line2D([0], [0], color='coral', lw=2, label='31d')
        ax.legend(handles=[lines_13, lines_31], fontsize=7, loc='lower right')

# Hide unused axes
for j in range(len(sorted_params), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Marginal HOTA vs parameter value at each age\n(dashed lines = age-specific optimum)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Linear age-scaling summary: which params are worth making age-dependent?
# "Worth it" = the HOTA improvement from using the age-specific opt vs the combined opt is meaningful

combined_opt_row = df[df['HOTA_combined'] == df['HOTA_combined'].max()].iloc[0]

print("Age-aware parameter formulas (apply at inference time):")
print("=" * 55)
print(f"  param(age) = slope × age + intercept\n")

# Only print params where the optimal value actually changes
age_dependent = [(p, results[p]) for p in sorted_params if abs(results[p]['delta']) > 1e-6]
stable = [(p, results[p]) for p in sorted_params if abs(results[p]['delta']) <= 1e-6]

for p, r in age_dependent:
    print(f"  {p}(age) = {r['slope']:.4f} × age + {r['intercept']:.4f}")
    print(f"           [13d→{r['opt_13']:.3f}, 31d→{r['opt_31']:.3f}]")

print(f"\nStable params (same optimum at all ages): {[p for p, _ in stable]}")
print(f"\nNote: linear model fitted on 2 points only (13d, 31d).")
print("Validate by running grid search on 28d data and checking if the predicted value lands near the new optimum.")
